# Plan B — Docking Molecular en Google Colab (GPU A100)

Este notebook ejecuta el **Experimento 1 del Plan B** (`biology/molecular_twin`):
- Prepara el receptor **Keap1 Kelch domain (`4IQK.pdb`)** usando el bolsillo cristalográfico real del ligando `IQK`.
- Corre docking molecular con **AutoDock Vina** para las 3 contramedidas radioprotectoras de `radiation-engine`:
  1. **N-Acetilcisteína (NAC)**
  2. **Vitamina C (L-Ascorbato)**
  3. **Vitamina E (Alfa-Tocoferol)**
- Extrae automáticamente las afinidades reales ($\Delta G$ en kcal/mol), convierte las poses a `.pdb`/`.sdf` para el visor 3D, actualiza `docking_catalog.json` y descarga un `.zip` listo para descomprimir en tu repo.

### Paso a paso antes de empezar
1. En Colab ve a **Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> GPU A100** (o T4 si no hay A100 disponible).
2. Ejecuta las celdas en orden de arriba hacia abajo.
3. En la **Celda 3** podrás subir los archivos de `biology/molecular_twin/structures/` y `biology/molecular_twin/data/docking_catalog.json` (o dejar que Colab los descargue automáticamente si no los subes).

In [ ]:
# 1. Verificar GPU y recursos del entorno
!nvidia-smi

In [ ]:
# 2. Instalar AutoDock Vina, OpenBabel y utilidades en Colab
!apt-get update -qq && apt-get install -y -qq autodock-vina openbabel > /dev/null
!pip install -q biopython
!vina --version
!obabel -V

### 3. Subir archivos locales (opcional) o descargar automáticamente
Puedes subir desde tu computador:
- `4IQK.pdb`, `1JEY.pdb`, `nac.sdf`, `ascorbate.sdf`, `tocopherol.sdf` (de `biology/molecular_twin/structures/`)
- `docking_catalog.json` (de `biology/molecular_twin/data/`)

Si presionas **Cancelar** o no subes alguno, la siguiente celda lo descargará automáticamente desde RCSB PDB y PubChem.

In [ ]:
import os
from pathlib import Path
from google.colab import files

WORK_DIR = Path("/content/molecular_twin_work")
STRUCT_DIR = WORK_DIR / "structures"
DATA_DIR = WORK_DIR / "data"
STRUCT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Selecciona los archivos a subir (o haz clic en Cancelar para descargarlos automáticamente):")
try:
    uploaded = files.upload()
    for fname, content in uploaded.items():
        if fname == "docking_catalog.json":
            (DATA_DIR / fname).write_bytes(content)
        else:
            (STRUCT_DIR / fname).write_bytes(content)
    print(f"Subidos {len(uploaded)} archivo(s).")
except Exception as e:
    print("Carga omitida o cancelada:", e)

In [ ]:
# 4. Completar archivos faltantes automáticamente desde RCSB PDB y PubChem
import urllib.request

RCSB_URLS = {
    "1JEY.pdb": "https://files.rcsb.org/download/1JEY.pdb",
    "4IQK.pdb": "https://files.rcsb.org/download/4IQK.pdb",
}
PUBCHEM_URLS = {
    "nac.sdf": "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/12035/SDF?record_type=3d",
    "ascorbate.sdf": "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/54670067/SDF?record_type=3d",
    "tocopherol.sdf": "https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/14985/SDF?record_type=3d",
}

for name, url in {**RCSB_URLS, **PUBCHEM_URLS}.items():
    dest = STRUCT_DIR / name
    if not dest.exists():
        print(f"Descargando {name}...")
        urllib.request.urlretrieve(url, dest)
    else:
        print(f"OK (ya existe): {name} ({dest.stat().st_size} bytes)")

In [ ]:
# 5. Preparar receptor Keap1 (4IQK) y calcular centro del bolsillo activo (ligando IQK)
import numpy as np

pdb_path = STRUCT_DIR / "4IQK.pdb"
clean_pdb = STRUCT_DIR / "4IQK_clean.pdb"
receptor_pdbqt = STRUCT_DIR / "4IQK_receptor.pdbqt"

ligand_coords = []
clean_lines = []
for line in pdb_path.read_text().splitlines():
    if line.startswith("HETATM") and line[17:20].strip() == "IQK":
        x = float(line[30:38])
        y = float(line[38:46])
        z = float(line[46:54])
        ligand_coords.append([x, y, z])
    elif line.startswith("ATOM"):  # Solo proteína limpia (sin aguas ni ligando previo)
        clean_lines.append(line)

clean_pdb.write_text("\n".join(clean_lines) + "\nEND\n")
center = np.mean(ligand_coords, axis=0)
cx, cy, cz = float(center[0]), float(center[1]), float(center[2])
print(f"Centro del bolsillo Keap1 (IQK): x={cx:.2f}, y={cy:.2f}, z={cz:.2f}")

!obabel {clean_pdb} -O {receptor_pdbqt} -xr -p 7.4

In [ ]:
# 6. Convertir ligandos a PDBQT y ejecutar AutoDock Vina (exhaustiveness=16)
import re
import subprocess

LIGANDS = {
    "keap1_nac": "nac",
    "keap1_ascorbate": "ascorbate",
    "keap1_tocopherol": "tocopherol",
}

docking_results = {}
for catalog_id, lig in LIGANDS.items():
    sdf_in = STRUCT_DIR / f"{lig}.sdf"
    pdbqt_in = STRUCT_DIR / f"{lig}.pdbqt"
    pdbqt_out = STRUCT_DIR / f"4IQK_keap1_{lig}_docked.pdbqt"
    pdb_out = STRUCT_DIR / f"4IQK_keap1_{lig}_docked.pdb"

    subprocess.run(["obabel", str(sdf_in), "-O", str(pdbqt_in), "-p", "7.4"], check=True)
    cmd = [
        "vina",
        "--receptor", str(receptor_pdbqt),
        "--ligand", str(pdbqt_in),
        "--center_x", f"{cx:.2f}",
        "--center_y", f"{cy:.2f}",
        "--center_z", f"{cz:.2f}",
        "--size_x", "24",
        "--size_y", "24",
        "--size_z", "24",
        "--exhaustiveness", "16",
        "--out", str(pdbqt_out),
    ]
    print(f"\n=== Docking {catalog_id} ({lig}) ===")
    proc = subprocess.run(cmd, capture_output=True, text=True, check=True)
    print(proc.stdout)

    # Extraer mejor afinidad (modo 1)
    match = re.search(r"^\s*1\s+(-?\d+\.\d+)", proc.stdout, re.MULTILINE)
    best_affinity = float(match.group(1)) if match else None
    docking_results[catalog_id] = {
        "affinity_kcal_mol": best_affinity,
        "structure_file": f"structures/4IQK_keap1_{lig}_docked.pdb",
    }
    subprocess.run(["obabel", str(pdbqt_out), "-O", str(pdb_out)], check=True)

print("\nResumen de afinidades reales (kcal/mol):")
for k, v in docking_results.items():
    print(f"  {k}: {v['affinity_kcal_mol']} kcal/mol -> {v['structure_file']}")

In [ ]:
# 7. Actualizar docking_catalog.json con las afinidades reales y rutas .pdb
import json

catalog_file = DATA_DIR / "docking_catalog.json"
if catalog_file.exists():
    catalog = json.loads(catalog_file.read_text())
    for item in catalog:
        cid = item["id"]
        if cid in docking_results and docking_results[cid]["affinity_kcal_mol"] is not None:
            item["bindingAffinityKcalMol"] = docking_results[cid]["affinity_kcal_mol"]
            item["structureFile"] = docking_results[cid]["structure_file"]
            item["source"] = "autodock_vina_colab_a100"
        elif cid == "ku70_ku80_simple_dsb":
            item["structureFile"] = "structures/1JEY.pdb"
    catalog_file.write_text(json.dumps(catalog, indent=2))
    print("docking_catalog.json actualizado con resultados reales.")
else:
    (DATA_DIR / "docking_results_summary.json").write_text(json.dumps(docking_results, indent=2))
    print("Guardado docking_results_summary.json (no se subió docking_catalog.json previo).")

In [ ]:
# 8. Empaquetar resultados y descargar .zip para descomprimir en biology/molecular_twin/
import shutil

zip_path = shutil.make_archive("/content/molecular_twin_a100_results", "zip", WORK_DIR)
print("Archivo listo:", zip_path)
files.download(zip_path)